In [44]:
from miscope import load_family

family = load_family("modulo_addition_1layer")
variant = family.get_variant(prime=109, seed=485, data_seed=598)
checkpoint_list = variant.get_available_checkpoints()

pinned_checkpoint = 1500
pinned_checkpoint_index = checkpoint_list.index(pinned_checkpoint)

variant, len(checkpoint_list)

(Variant(family='modulo_addition_1layer', name='p109_seed485_dseed598', state=analyzed),
 353)

In [ ]:
# Load weights
weights = variant.artifacts.load_epoch("parameter_snapshot", pinned_checkpoint)
W_E = weights["W_E"]        # (d_vocab, d_model)
W_U = weights["W_U"]        # (d_model, d_vocab)
W_Q = weights["W_Q"]        # (n_heads, d_model, d_head)
W_K = weights["W_K"]        # (n_heads, d_model, d_head)
W_O = weights["W_O"]        # (n_heads, d_head, d_model)
W_V = weights["W_V"]        # (n_heads, d_model, d_head)
W_in = weights["W_in"]      # (d_model, d_mlp)
W_out = weights["W_out"]    # (d_mlp, d_model)

head_count = W_Q.shape[0]
W_QKT = []
W_OV = []
QK_circuit = []
OV_circuit = []

for head in range(head_count):
    W_Q_h, W_K_h = W_Q[head, :], W_K[head, :]

    w_qkt_h = W_Q_h @ W_K_h.T       # (d_model, d_head) @ (d_head, d_model)
    w_ov_h =  W_V[head] @ W_O[head] # (d_model, d_head) @ (d_head, d_model)

    W_QKT.append(w_qkt_h) 
    W_OV.append(w_ov_h)

    #print(f"w_qkt_h.shape: {w_qkt_h.shape}")
    #print(f"w_ov_h.shape: {w_ov_h.shape}")

    QK_circuit.append(W_E @ W_QKT[head] @ W_E.T)    # (d_vocab, d_model) @ (d_model, d_model) @ (d_model, d_vocab)
    OV_circuit.append(W_U.T @ W_OV[head] @ W_E.T)   # (d_vocab, d_model) @ (d_model, d_model) @ (d_model, d_vocab)

print(f"W_E.shape: {W_E.shape}\nW_U.shape: {W_U.shape}")
print(f"W_Q.shape: {W_Q.shape}\nW_K.shape: {W_K.shape}")
print(f"W_O.shape: {W_O.shape}\nW_V.shape: {W_V.shape}")
print(f"QK_circuit[0].shape: {QK_circuit[0].shape}")
print(f"OV_circuit[0].shape: {OV_circuit[0].shape}")
print(f"W_in.shape: {W_in.shape}\nW_out.shape: {W_out.shape}")
print(f"Head Count: {head_count}")



W_E.shape: (110, 128)
W_U.shape: (128, 109)
W_Q.shape: (4, 128, 32)
W_K.shape: (4, 128, 32)
W_O.shape: (4, 32, 128)
W_V.shape: (4, 128, 32)
QK_circuit[0].shape: (110, 110)
OV_circuit[0].shape: (109, 110)
W_in.shape: (128, 512)
W_out.shape: (512, 128)
Head Count: 4


In [ ]:
# Load activations
